# Agent AI Bootcamp

Code from the section notes goes into cells below. Keys are read from `.env` in the project root.

In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os

# Groq (OpenAI-compatible). To use OpenAI instead, replace with: openai_client = OpenAI()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

Quick API check. Run this after putting your real Groq key in `.env`; a 401 error means the key is wrong or still the placeholder.

In [2]:
response = openai_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Reply with the single word: ok"}],
)
print(response.choices[0].message.content)

ok


## 03 RAG: Retrieval-Augmented Generation

Course notes use OpenAI `responses.create` with `gpt-5.4-mini`. We use Groq, so the same idea is written with `chat.completions` and `openai/gpt-oss-120b`.

In [3]:
MODEL = "openai/gpt-oss-120b"

def llm(prompt):
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

In [4]:
llm("Hey, what's up?")

"Hey! I'm doing great—thanks for asking. How can I help you today?"

### Plain LLM lacks our data
A course-specific question gets a generic answer: the model knows nothing about Zoomcamp enrollment.

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Absolutely! 🎉 You can enroll in the course at any time—our program is designed to be flexible so you can start whenever you’re ready.

### How to Get Started
1. **Visit the enrollment page** – Head over to the course’s website and click the “Enroll Now” button.  
2. **Create an account** – If you don’t already have one, you’ll need to sign up with your name, email, and a password.  
3. **Choose a payment plan** – We offer several options (full payment, monthly installments, or a free trial period for the first module).  
4. **Complete the short questionnaire** – This helps us tailor the learning path to your current skill level and goals.  
5. **Start the first lesson** – Once you’ve confirmed your enrollment, you’ll get immediate access to the introductory module, which includes a welcome video, syllabus overview, and a quick “getting‑started” tutorial.

### Things to Keep in Mind
- **Prerequisites** – The course assumes basic familiarity with [relevant background, e.g., programming c

### Adding context manually
Paste FAQ entries into the prompt. This is RAG done by hand.

In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
answer = llm(prompt)
print(answer)

Yes—you can still join the course. You can start learning and submit homework right away (the registration list isn’t enforced). Just keep in mind that if you want to earn a certificate, you’ll need to submit your final project before the submission window closes.


### RAG = search + prompt + LLM

```
User question -> search(knowledge base) -> top documents -> build_prompt(question, docs) -> llm(prompt) -> answer
```

The LLM only sees the documents we hand it, so answers are grounded in our data. Retrieval quality decides answer quality. `search` and `build_prompt` are implemented in the next sections (dataset + minsearch); the skeleton below is the whole architecture.

In [9]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

# search() and build_prompt() come in the next section

## 04 The Course FAQ Dataset
FAQ data from DataTalks.Club as JSON. This is our RAG knowledge base.

In [10]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()
len(courses_raw), courses_raw[0]

(6,
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 482})

In [11]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1456

In [12]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [13]:
from collections import Counter
Counter(d["course"] for d in documents)

Counter({'machine-learning-zoomcamp': 482,
         'data-engineering-zoomcamp': 398,
         'mlops-zoomcamp': 249,
         'llm-zoomcamp': 161,
         'stock-markets-analytics-zoomcamp': 93,
         'ai-dev-tools-zoomcamp': 73})

## 05 Search with minsearch
Text fields are tokenized and ranked; keyword fields are exact-match filters.

**Windows note:** real `minsearch` fails to import here (Smart App Control blocks scipy `_ckdtree` DLL). The cell falls back to `minsearch_lite.py`, same API, standard library only.

In [14]:
# Course code: from minsearch import Index
# Smart App Control blocks scipy DLLs that minsearch needs (via scikit-learn),
# so we use a pure-Python drop-in with the same API: notebooks/minsearch_lite.py
try:
    from minsearch import Index
except ImportError:
    from minsearch_lite import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [15]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Does the course certificate show the number of course hours?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?']

### Boosting fields

In [16]:
results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)
[(doc["course"], doc["question"]) for doc in results]

[('llm-zoomcamp', 'I just discovered the course. Can I still join?'),
 ('machine-learning-zoomcamp',
  'The course has already started. Can I still join it?'),
 ('mlops-zoomcamp',
  'Course - Can I still join the course after the start date?'),
 ('data-engineering-zoomcamp',
  'Course: Can I still join the course after the start date?'),
 ('data-engineering-zoomcamp', 'Course: When does the course start?')]

### Filtering by course

In [17]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Course: How do I start?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?']

### search() function: first RAG component

In [18]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

search_results = search(question)
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Does the course certificate show the number of course hours?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?']

### Lexical search limitation
Same meaning, different words. Text search only matches words.

In [19]:
[doc["question"] for doc in search("Is it possible to enroll late?")]

['Is it a group project?',
 "My dataset is copyrighted and I can't publish it. Can I still use it for the project?",
 'uv says Failed to hardlink files in Codespaces. Is it an error?',
 'My knowledge base file is too large for the repository. How do I share it with reviewers?',
 'What happens to code saved in Codespaces if I do not commit it?']

## 06 Building the Prompt
Instructions (system prompt) stay fixed. The user prompt changes per request: question + retrieved context.

In [20]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [21]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [22]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [23]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [24]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI w

## 07 The LLM
Course uses OpenAI `responses.create`. Groq only exposes `chat.completions`, so the response shape differs:

| OpenAI Responses | Groq chat.completions |
|---|---|
| `input=prompt` | `messages=[{"role": "user", "content": prompt}]` |
| `response.output_text` | `response.choices[0].message.content` |
| `usage.input_tokens` / `output_tokens` | `usage.prompt_tokens` / `completion_tokens` |

In [25]:
response = openai_client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}]
)

### Exploring the response

In [26]:
response.choices[0]

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Yes—you can still join the LLM\u202fZoomcamp. Feel free to start working through the lessons and doing the homework right away.  \n\n**Just keep in mind:** if you want to earn the official course certificate, you’ll need to submit your capstone project (and complete the required peer reviews) before the current live cohort closes its submission window. Otherwise you can still learn at your own pace with no penalty. Happy learning!', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='The user asks: "I just discovered the course. Can I join now?" The context includes a Q&A: "I just discovered the course. Can I still join? A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions." So answer that. Provide guidance.'))

In [27]:
response.choices[0].message.content

'Yes—you can still join the LLM\u202fZoomcamp. Feel free to start working through the lessons and doing the homework right away.  \n\n**Just keep in mind:** if you want to earn the official course certificate, you’ll need to submit your capstone project (and complete the required peer reviews) before the current live cohort closes its submission window. Otherwise you can still learn at your own pace with no penalty. Happy learning!'

In [28]:
response.usage

CompletionUsage(completion_tokens=165, prompt_tokens=571, total_tokens=736, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=69, rejected_prediction_tokens=None, text_tokens=None), prompt_tokens_details=None, queue_time=0.334814264, prompt_time=0.026761047, completion_time=0.352019651, total_time=0.378780698)

### Calculating the price
Groq `openai/gpt-oss-120b`: $0.15 input / $0.60 output per million tokens (console.groq.com/docs/model/openai/gpt-oss-120b, Oct 2026). On the free tier you pay nothing; this shows what it would cost.

In [29]:
input_price = 0.15 / 1_000_000
output_price = 0.60 / 1_000_000

cost = (
    response.usage.prompt_tokens * input_price +
    response.usage.completion_tokens * output_price
)

cost

0.00018465

### Message history: developer + user

In [30]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.chat.completions.create(
    model=MODEL,
    messages=message_history
)
print(response.choices[0].message.content)

Yes—you can still join the course. Just keep in mind that if you’d like to earn a certificate, you’ll need to submit your capstone project while the course is still accepting submissions.


### The llm() function (replaces the Lab 03 version)

In [31]:
def llm(instructions, user_prompt, model=MODEL):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.chat.completions.create(
        model=model,
        messages=message_history
    )

    return response.choices[0].message.content

### Full RAG

In [32]:
def rag(query, model=MODEL):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [33]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes—you can still join the course. If you want to earn a certificate, just be sure to submit your capstone project while the program is still accepting submissions.


In [34]:
print(rag("How do I get a certificate?"))

To receive a certificate you must:

1. **Enroll in a live cohort** – certificates are only issued for students who finish the course during an active (“live”) cohort.  
2. **Complete the Capstone project** – submit a passing version of the project. Homework is optional and does not affect eligibility.  
3. **Finish all assigned peer‑reviews** – after you submit the project, you’ll automatically receive review assignments. Complete them within the peer‑review window.  

You can work through the lectures and even prepare your project in self‑paced mode, but the final project submission and the peer‑review tasks must occur while a live cohort is open.

Once your last project attempt and its peer reviews are finished, the instructors generate certificates in one batch for the whole cohort. You don’t need to request it; just make sure the name on your profile in the course‑management platform is correct. When the certificate is ready you’ll receive an email (check your spam folder if you do

## 08 RAG Helper
Reusable code now lives in `notebooks/ingest.py` (load data, build index) and `notebooks/rag_helper.py` (`RAGBase` class). The cells below start fresh from imports, as you would in a new notebook.

In [35]:
from dotenv import load_dotenv
load_dotenv()

import os
from openai import OpenAI
from ingest import load_faq_data, build_index
from rag_helper import RAGBase

documents = load_faq_data()
index = build_index(documents)

# Course: openai_client = OpenAI()
groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

assistant = RAGBase(
    index=index,
    llm_client=groq_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes—you can still join the course. Just keep in mind that if you want to earn a certificate, you’ll need to submit your capstone project before the submission window closes.


### Custom instructions

In [36]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=groq_client,
    instructions=custom_instructions,
)

In [37]:
print(assistant.rag("How do I get a certificate?"))

To receive a certificate you must:

1. **Be enrolled in a live cohort** – certificates are only issued to learners who finish the course with a “live” cohort; a fully self‑paced track does not qualify.  

2. **Complete the capstone project** – submit a passing version of the project. You can work on the material and draft the project in self‑paced mode, but the final submission must occur while a live cohort is open and accepting projects.  

3. **Finish the required peer reviews** – after you submit the project you will automatically receive peer‑review assignments. Complete those reviews before the peer‑review deadline. No extra sign‑up is needed.  

4. **(Optional) Verify your name** – before the batch is generated, make sure the name shown in your profile on the course management platform is exactly how you want it printed on the certificate.  

5. **Wait for batch processing** – certificates are generated manually in one batch for all graduates of the cohort. Once the last project

In [38]:
print(assistant.rag("Can I still join the course after it started?"))

Yes—you can still join the course even after it has begun. Just keep in mind that if you want to earn a certificate, you’ll need to submit your capstone project before the submission window closes. (Homework is optional and not required for the certificate.)


### Other course, same class
Only the constructor argument changes.

In [39]:
mlops_assistant = RAGBase(index=index, llm_client=groq_client, course="mlops-zoomcamp")
print(mlops_assistant.rag("Can I still join the course after it started?"))

Yes. You can join the course even after it has started. Registration isn’t required, and you’re still allowed to submit the homework assignments as long as the submission form for each assignment remains open. Just keep an eye on the deadlines for the final projects, since those do have fixed due dates.
